In [1]:
!pip install -q transformers datasets scikit-learn

In [2]:
import os
import re
import pandas as pd
import numpy as np
import torch
from torch import nn
import torch.nn.functional as F

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, balanced_accuracy_score
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.utils.class_weight import compute_class_weight

from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset

# =========================
# 공통 설정
# =========================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

MODEL_NAME = "klue/bert-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

MAX_LENGTH = 256

# AE: 1 = 공감형(E)
AXIS_THRESHOLDS = {
    "AE": 0.56,
}

print("device:", device)
print("MODEL_NAME:", MODEL_NAME)
print("MAX_LENGTH:", MAX_LENGTH)
print("AXIS_THRESHOLDS:", AXIS_THRESHOLDS)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/425 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/289 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

device: cuda
MODEL_NAME: klue/bert-base
MAX_LENGTH: 256
AXIS_THRESHOLDS: {'AE': 0.56}


In [3]:
def train_axis_model(csv_path, save_dir):
    df = pd.read_csv(csv_path).dropna()
    df.columns = ['review_text', 'label']
    df['label'] = df['label'].astype(int)

    # 저장 폴더 미리 생성
    os.makedirs(save_dir, exist_ok=True)

    # ✅ 1. train/validation split을 여기서 한 번만 만든다
    train_df, val_df = train_test_split(
        df,
        test_size=0.2,
        random_state=42,
        stratify=df['label']   # A/E 비율이 train과 val에 비슷하게 들어가도록 함
    )

    train_df = train_df.reset_index(drop=True)
    val_df = val_df.reset_index(drop=True)

    # ✅ 2. 나중에 오분류 분석할 때 같은 검증셋을 쓰기 위해 저장
    train_df.to_csv(f"{save_dir}/train_split.csv", index=False, encoding="utf-8-sig")
    val_df.to_csv(f"{save_dir}/val_split.csv", index=False, encoding="utf-8-sig")

    print(f"학습셋 크기: {len(train_df)}개")
    print(f"검증셋 크기: {len(val_df)}개")
    print("검증셋 라벨 분포")
    print(val_df['label'].value_counts().sort_index())

    # ✅ 3. class weight는 전체 df가 아니라 train_df 기준으로 계산하는 게 더 안전함
    class_weights = compute_class_weight(
        'balanced',
        classes=np.unique(train_df['label']),
        y=train_df['label']
    )
    weights_tensor = torch.tensor(class_weights, dtype=torch.float32).to(device)

    # ✅ 4. sklearn으로 나눈 train_df, val_df를 HuggingFace Dataset으로 변환
    train_dataset = Dataset.from_pandas(train_df)
    val_dataset = Dataset.from_pandas(val_df)

    def tokenize_fn(x):
        return tokenizer(
            x["review_text"],
            truncation=True,
            padding="max_length",
            max_length=MAX_LENGTH
        )

    tokenized_train = train_dataset.map(tokenize_fn, batched=True)
    tokenized_val = val_dataset.map(tokenize_fn, batched=True)

    def compute_metrics(pred):
        labels = pred.label_ids
        preds = pred.predictions.argmax(-1)

        return {
            "accuracy": accuracy_score(labels, preds),
            "f1_macro": f1_score(labels, preds, average="macro"),
            "f1_binary": f1_score(labels, preds, average="binary"),
            "balanced_accuracy": balanced_accuracy_score(labels, preds)
        }

    class CustomTrainer(Trainer):
        def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
            labels = inputs.pop("labels")
            outputs = model(**inputs)
            loss_fct = nn.CrossEntropyLoss(weight=weights_tensor)
            loss = loss_fct(outputs.logits.view(-1, 2), labels.view(-1))
            return (loss, outputs) if return_outputs else loss

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=2
    ).to(device)

    args = TrainingArguments(
      output_dir=save_dir,
      eval_strategy="epoch",
      save_strategy="epoch",
      learning_rate=2e-5,
      num_train_epochs=4,
      per_device_train_batch_size=16,
      load_best_model_at_end=True,
      metric_for_best_model="f1_macro",
      greater_is_better=True
  )

    trainer = CustomTrainer(
        model=model,
        args=args,
        train_dataset=tokenized_train,
        eval_dataset=tokenized_val,
        compute_metrics=compute_metrics
    )

    trainer.train()

    trainer.save_model(save_dir)
    tokenizer.save_pretrained(save_dir)

    print(f"✅ 모델 저장 완료: {save_dir}")
    print(f"✅ 검증셋 저장 완료: {save_dir}/val_split.csv")


In [4]:
# =========================
# AE 모델만 학습
# =========================
AE_CSV_PATH = "/content/train_AE.csv"
AE_SAVE_DIR = "./model_axis2_AE"

train_axis_model(
    AE_CSV_PATH,
    AE_SAVE_DIR
)


학습셋 크기: 2400개
검증셋 크기: 600개
검증셋 라벨 분포
label
0    300
1    300
Name: count, dtype: int64


Map:   0%|          | 0/2400 [00:00<?, ? examples/s]

Map:   0%|          | 0/600 [00:00<?, ? examples/s]

model.safetensors:   0%|          | 0.00/445M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: klue/bert-base
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on you

... (4 outputs left out)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ 모델 저장 완료: ./model_axis2_AE
✅ 검증셋 저장 완료: ./model_axis2_AE/val_split.csv


In [5]:
# =========================
# AE 모델 로드
# =========================
model_ae = AutoModelForSequenceClassification.from_pretrained("./model_axis2_AE").to(device)
model_ae.eval()

AE_THRESHOLD = AXIS_THRESHOLDS["AE"]

print("✅ AE 모델 로드 완료")
print("AE_THRESHOLD:", AE_THRESHOLD)


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

✅ AE 모델 로드 완료
AE_THRESHOLD: 0.56


In [8]:
# =========================
# AE 문장 단위 단서 기반 보정 규칙
# 기존 방식 유지: 모델 예측 + 문장 단위 cue 분석 + A/E 보정
# =========================

A_CUES = [
    "이 책은", "작품은", "작가는",
    "구성", "전개", "구조", "흐름", "설명", "정리",
    "사례", "개념", "논리", "장점", "단점",
    "주제", "메시지", "배치", "결말", "갈등",
    "관계 변화", "체계적", "안정적", "자연스럽게",
    "이해하기 쉽게", "전체적으로", "전체 구성"
]

E_CUES = [
    "나는", "내가", "나에게", "내 상황", "내 이야기",
    "마음", "감정", "공감", "위로", "먹먹",
    "울었다", "찡했다", "슬펐다", "아팠다",
    "떠올랐다", "겹쳐", "찔렸다", "와닿았다"
]


def split_sentences(text):
    sentences = re.split(r'(?<=[.!?。！？])\s+', text.strip())
    sentences = [s.strip() for s in sentences if s.strip()]
    return sentences if sentences else [text]


def count_cues(text, cues):
    return sum(1 for cue in cues if cue in text)


def get_ae_base_prediction(text):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH
    ).to(device)

    with torch.no_grad():
        outputs = model_ae(**inputs)

    probs = F.softmax(outputs.logits, dim=-1)[0]
    prob_A = probs[0].item()
    prob_E = probs[1].item()
    base_pred = 1 if prob_E >= AE_THRESHOLD else 0

    return base_pred, prob_A, prob_E


def analyze_ae_sentences(text):
    sentences = split_sentences(text)
    rows = []

    for idx, sent in enumerate(sentences, start=1):
        a_count = count_cues(sent, A_CUES)
        e_count = count_cues(sent, E_CUES)

        if a_count > e_count:
            cue_label = "분석형(A)"
        elif e_count > a_count:
            cue_label = "공감형(E)"
        else:
            cue_label = "중립/불명확"

        rows.append({
            "문장번호": idx,
            "문장": sent,
            "A_cue_count": a_count,
            "E_cue_count": e_count,
            "문장단서판단": cue_label,
        })

    return pd.DataFrame(rows)


def predict_ae_with_context_rule(text):
    """
    A/E 전용 보정 예측.
    기본 모델은 그대로 쓰되,
    책 구조 평가 문장이 다수이고 감정 문장이 일부인 경우 A로 보정한다.
    """

    # 1. 전체 리뷰 기준 모델 예측
    base_pred, prob_A, prob_E = get_ae_base_prediction(text)

    # 2. 문장 단위 cue 분석
    sentence_df = analyze_ae_sentences(text)

    a_sentence_count = (sentence_df["문장단서판단"] == "분석형(A)").sum()
    e_sentence_count = (sentence_df["문장단서판단"] == "공감형(E)").sum()

    total_a_cues = count_cues(text, A_CUES)
    total_e_cues = count_cues(text, E_CUES)

    # 3. 보정 규칙

    # 모델은 E라고 했지만 → A로 보정
    if base_pred == 1:
        if (
            a_sentence_count >= 2
            and total_a_cues >= total_e_cues + 2
            and total_a_cues >= 4
        ):
            corrected_pred = 0
            correction_reason = "A/E 보정: 책 구조·전개·구성 평가가 리뷰 중심"
        else:
            corrected_pred = base_pred
            correction_reason = "모델 예측 유지"

    # 모델은 A라고 했지만 → E로 보정
    elif base_pred == 0:
        if (
            e_sentence_count >= 2
            and total_e_cues >= total_a_cues + 2
            and total_e_cues >= 4
        ):
            corrected_pred = 1
            correction_reason = "A/E 보정: 감정·경험 중심 서술이 리뷰 중심"
        else:
            corrected_pred = base_pred
            correction_reason = "모델 예측 유지"

    else:
        corrected_pred = base_pred
        correction_reason = "모델 예측 유지"


    conf = prob_E if corrected_pred == 1 else prob_A

    return {
        "pred": corrected_pred,
        "label": "분석형(A)" if corrected_pred == 0 else "공감형(E)",
        "confidence": conf,
        "prob_A": prob_A,
        "prob_E": prob_E,
        "base_pred": base_pred,
        "base_label": "분석형(A)" if base_pred == 0 else "공감형(E)",
        "correction_reason": correction_reason,
        "a_sentence_count": int(a_sentence_count),
        "e_sentence_count": int(e_sentence_count),
        "total_a_cues": int(total_a_cues),
        "total_e_cues": int(total_e_cues),
        "sentence_df": sentence_df,
    }


In [9]:
# =========================
# 보기 좋은 단일 리뷰 출력 함수
# =========================

def display_ae_prediction(text):
    result = predict_ae_with_context_rule(text)

    summary_df = pd.DataFrame([{
        "최종 예측": result["label"],
        "최종 확신도": f"{result['confidence'] * 100:.1f}%",
        "분석형(A) 확률": f"{result['prob_A'] * 100:.1f}%",
        "공감형(E) 확률": f"{result['prob_E'] * 100:.1f}%",
        "모델 기본 예측": result["base_label"],
        "보정 사유": result["correction_reason"],
        "분석형 문장 수": result["a_sentence_count"],
        "공감형 문장 수": result["e_sentence_count"],
        "A 단서 수": result["total_a_cues"],
        "E 단서 수": result["total_e_cues"],
    }])

    print("=" * 80)
    print("입력 리뷰")
    print("=" * 80)
    print(text.strip())

    print("\n" + "=" * 80)
    print("AE 최종 예측 요약")
    print("=" * 80)
    display(summary_df)

    print("\n" + "=" * 80)
    print("문장 단위 분석")
    print("=" * 80)
    display(result["sentence_df"])

    return result


In [18]:
# =========================
# 테스트 리뷰
# =========================

test_review = """
책의 전개 방식은 비교적 단순하지만, 핵심 개념을 반복적으로 강조하는 구조가 눈에 띈다. 이런 구성 덕분에 내용을 이해하기는 어렵지 않았다. 다만 읽다 보니 내 경험이 떠올라 약간 복잡한 감정이 들기도 했다.
"""

result = display_ae_prediction(test_review)


입력 리뷰
책의 전개 방식은 비교적 단순하지만, 핵심 개념을 반복적으로 강조하는 구조가 눈에 띈다. 이런 구성 덕분에 내용을 이해하기는 어렵지 않았다. 다만 읽다 보니 내 경험이 떠올라 약간 복잡한 감정이 들기도 했다.

AE 최종 예측 요약


,최종 예측,최종 확신도,분석형(A) 확률,공감형(E) 확률,모델 기본 예측,보정 사유,분석형 문장 수,공감형 문장 수,A 단서 수,E 단서 수
0,분석형(A),28.5%,28.5%,71.5%,공감형(E),A/E 보정: 책 구조·전개·구성 평가가 리뷰 중심,2,1,4,1



문장 단위 분석


,문장번호,문장,A_cue_count,E_cue_count,문장단서판단
0,1,"책의 전개 방식은 비교적 단순하지만, 핵심 개념을 반복적으로 강조하는 구조가 눈에 띈다.",3,0,분석형(A)
1,2,이런 구성 덕분에 내용을 이해하기는 어렵지 않았다.,1,0,분석형(A)
2,3,다만 읽다 보니 내 경험이 떠올라 약간 복잡한 감정이 들기도 했다.,0,1,공감형(E)


In [10]:
# =========================
# AE 검증셋 성능 / 오분류 요약
# =========================

def predict_ae_batch(texts, batch_size=32, max_length=MAX_LENGTH):
    model_ae.eval()

    preds = []
    probs_A = []
    probs_E = []
    confidences = []

    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]

        inputs = tokenizer(
            batch,
            return_tensors="pt",
            truncation=True,
            padding=True,
            max_length=max_length
        ).to(device)

        with torch.no_grad():
            logits = model_ae(**inputs).logits

        prob = F.softmax(logits, dim=-1).cpu()
        prob_A = prob[:, 0]
        prob_E = prob[:, 1]

        pred = (prob_E >= AE_THRESHOLD).long()
        confidence = torch.where(pred == 1, prob_E, prob_A)

        preds.extend(pred.tolist())
        probs_A.extend(prob_A.tolist())
        probs_E.extend(prob_E.tolist())
        confidences.extend(confidence.tolist())

    return preds, probs_A, probs_E, confidences


def analyze_ae_validation(val_csv_path="./model_axis2_AE/val_split.csv", save_dir="/content"):
    print("\n" + "=" * 70)
    print("AE 축 성능 / 오분류 요약")
    print("=" * 70)

    val_df = pd.read_csv(val_csv_path)
    val_df["label"] = val_df["label"].astype(int)

    label_counts = val_df["label"].value_counts().sort_index()
    print("\n[1] 검증셋 라벨 분포")
    print(f"  분석형(A=0): {label_counts.get(0, 0)}개")
    print(f"  공감형(E=1): {label_counts.get(1, 0)}개")
    print(f"  전체: {len(val_df)}개")

    texts = val_df["review_text"].tolist()
    preds, probs_A, probs_E, confidences = predict_ae_batch(texts)

    val_df["pred"] = preds
    val_df["prob_A"] = probs_A
    val_df["prob_E"] = probs_E
    val_df["confidence"] = confidences
    val_df["correct"] = val_df["label"] == val_df["pred"]

    y_true = val_df["label"]
    y_pred = val_df["pred"]

    acc = accuracy_score(y_true, y_pred)
    macro_f1 = f1_score(y_true, y_pred, average="macro")

    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])

    print("\n[2] 전체 성능")
    print(f"  Accuracy : {acc:.4f}")
    print(f"  Macro F1 : {macro_f1:.4f}")

    print("\n[3] Confusion Matrix")
    print("       예측 A    예측 E")
    print(f"실제 A   {cm[0,0]:>5}    {cm[0,1]:>5}")
    print(f"실제 E   {cm[1,0]:>5}    {cm[1,1]:>5}")

    print("\n[4] 오분류 방향")
    print(f"  분석형(A) → 공감형(E): {cm[0,1]}개")
    print(f"  공감형(E) → 분석형(A): {cm[1,0]}개")

    print("\n[5] 상세 리포트")
    print(classification_report(
        y_true,
        y_pred,
        labels=[0, 1],
        target_names=["분석형(A)", "공감형(E)"],
        digits=4
    ))

    os.makedirs(save_dir, exist_ok=True)
    result_path = f"{save_dir}/prediction_result_AE.csv"
    val_df.to_csv(result_path, index=False, encoding="utf-8-sig")

    print("\n[6] 저장 완료")
    print(f"  전체 예측 결과: {result_path}")

    return val_df


ae_result_df = analyze_ae_validation()



AE 축 성능 / 오분류 요약

[1] 검증셋 라벨 분포
  분석형(A=0): 300개
  공감형(E=1): 300개
  전체: 600개

[2] 전체 성능
  Accuracy : 0.8083
  Macro F1 : 0.8072

[3] Confusion Matrix
       예측 A    예측 E
실제 A     265       35
실제 E      80      220

[4] 오분류 방향
  분석형(A) → 공감형(E): 35개
  공감형(E) → 분석형(A): 80개

[5] 상세 리포트
              precision    recall  f1-score   support

      분석형(A)     0.7681    0.8833    0.8217       300
      공감형(E)     0.8627    0.7333    0.7928       300

    accuracy                         0.8083       600
   macro avg     0.8154    0.8083    0.8072       600
weighted avg     0.8154    0.8083    0.8072       600


[6] 저장 완료
  전체 예측 결과: /content/prediction_result_AE.csv


In [11]:
# =========================
# 여러 리뷰를 한 번에 예측하고 표로 보기
# =========================

def predict_ae_reviews_to_table(reviews):
    rows = []
    for i, text in enumerate(reviews, start=1):
        result = predict_ae_with_context_rule(text)
        rows.append({
            "번호": i,
            "리뷰": text.strip().replace("\n", " ")[:120],
            "최종 예측": result["label"],
            "분석형(A) 확률": result["prob_A"],
            "공감형(E) 확률": result["prob_E"],
            "보정 사유": result["correction_reason"],
            "분석형 문장 수": result["a_sentence_count"],
            "공감형 문장 수": result["e_sentence_count"],
        })
    return pd.DataFrame(rows)

sample_reviews = [
    "이 책은 인간관계의 원리를 사례를 통해 체계적으로 설명한다. 각 장의 구성도 명확하고 핵심 개념도 잘 정리되어 있다.",
    "읽는 내내 내가 겪었던 일이 떠올라서 마음이 아팠다. 주인공의 감정이 너무 잘 느껴져서 깊이 공감했다.",
    "이 책은 인간관계의 원리를 사례를 통해 체계적으로 설명한다. 각 장의 구성이 명확하다. 다만 내 경험이 떠올라 조금 씁쓸했다.",
]

sample_result_df = predict_ae_reviews_to_table(sample_reviews)
display(sample_result_df)


,번호,리뷰,최종 예측,분석형(A) 확률,공감형(E) 확률,보정 사유,분석형 문장 수,공감형 문장 수
0,1,이 책은 인간관계의 원리를 사례를 통해 체계적으로 설명한다. 각 장의 구성도 명확하...,분석형(A),0.993935,0.006065,모델 예측 유지,2,0
1,2,읽는 내내 내가 겪었던 일이 떠올라서 마음이 아팠다. 주인공의 감정이 너무 잘 느껴...,공감형(E),0.012590,0.987410,모델 예측 유지,0,2
2,3,이 책은 인간관계의 원리를 사례를 통해 체계적으로 설명한다. 각 장의 구성이 명확하...,분석형(A),0.403583,0.596417,A/E 보정: 책 구조·전개·구성 평가가 리뷰 중심,2,0
